# Data Preparation for EDA and clustering steps

In [1]:
import importlib
import sys, os

# Ensure project root is on path
sys.path.insert(0, os.path.abspath(".."))

import src.code.io_utils as io

importlib.reload(io)

import string
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import RobustScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.3f}".format)

# Path 
ABT_OUT_PATH  = "../data/prepared/abt.parquet"

In [2]:
customer = io.load(ABT_OUT_PATH)
customer.head(5)

[LOAD] ../data/prepared/abt.parquet | shape: (148729, 88)


,CONTRIB,N_CONTRACTS,FIRST_DCREAT,LAST_DCREAT,LAST_DPOS,LAST_DATFIN,FIRST_D1FIN,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,TOTAL_MENSALIDADE,TOTAL_CRD,TOTAL_SREC,TOTAL_RN,TOTAL_RD,LAST_RISK,MAX_RISKA,MIN_RESSO,MAX_RESSO,MEDIAN_RESSO,CSP,NBENF,...,MONTVENC_CP_MEDIAN,MONTVENC_AUTO_MIN,MONTVENC_AUTO_MAX,MONTVENC_AUTO_MEDIAN,MONTVENC_HT_MIN,MONTVENC_HT_MAX,MONTVENC_HT_MEDIAN,DIVIDAS_CL_MIN,DIVIDAS_CL_MAX,DIVIDAS_CL_MEDIAN,DIVIDAS_CP_MIN,DIVIDAS_CP_MAX,DIVIDAS_CP_MEDIAN,DIVIDAS_AUTO_MIN,DIVIDAS_AUTO_MAX,DIVIDAS_AUTO_MEDIAN,DIVIDAS_HT_MIN,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,sys_data_procura,kp_sqe_enc,ks_score_tier,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_SITFAM,sdem_HABITAT,sdem_age
0,00008246f87bcc3c17b90629bb183fe2e58795176310f0...,2,2024-06-25,2024-09-13,2025-11-17,2024-09-30,2024-06-27,84.000,120.000,102.000,3.000,13.000,8.000,19.000,19.000,19.000,24000.000,24000.000,438.761,0.000,0.000,0.000,0.000,0,0.000,1513.466,1513.466,1513.466,80.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,8000.000,0.000,1.650,3701.730,978.250,0.000,58633.860,15934.050,0.000,0.000,0.000,2025-02-05,7.000,2.000,10.000,1.000,2.000,-438.900,10.000,S,F,33.000
1,0000ab2116257783438c70ff85a3e98f2d4194ebe53434...,1,2018-03-29,2018-03-29,2018-04-16,2018-04-16,2018-04-16,120.000,120.000,120.000,91.000,91.000,91.000,91.000,91.000,91.000,20000.000,20000.000,347.447,8115.247,0.000,0.000,0.000,0,0.000,1113.258,1113.258,1113.258,80.000,1.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,20308.940,69754.560,24983.220,5499.880,21274.830,7115.460,0.000,0.000,0.000,0.000,0.000,0.000,2025-04-09,3.000,1.000,10.000,1.000,0.000,1166.900,10.000,C,P,52.000
2,0000c74654405ec1da4dbdcd00b86e397954043965d98e...,2,2001-09-21,2001-09-27,2014-06-27,2001-10-04,2001-09-21,60.000,60.000,60.000,21.000,21.000,21.000,21.000,21.000,21.000,19453.110,19453.110,608.914,0.000,0.000,2.000,24.000,-9223372036854775808,0.000,678.483,678.483,678.483,60.000,2.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None,NaN
3,0000e359b15a80ba12c7f60c0fe06ffc68621c87f13a4f...,1,2019-01-28,2019-01-28,2022-12-28,2019-02-04,2019-02-04,72.000,72.000,72.000,34.000,34.000,34.000,34.000,34.000,34.000,2500.000,2500.000,56.018,0.000,0.000,1.000,1.000,210210000000,0.000,838.186,838.186,838.186,91.000,0.000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaN,3.000,0.000,0.000,590.333,3.000,S,A,69.000
4,0000f858346061c53064586a3347b34659565a6712d004...,1,2019-09-23,2019-09-23,2019-09-30,2019-09-30,2019-09-30,84.000,84.000,84.000,74.000,74.000,74.000,74.000,74.000,74.000,5000.000,5000.000,100.074,883.500,0.000,0.000,0.000,0,0.000,1314.144,1314.144,1314.144,80.000,2.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1019.130,2957.280,2114.780,1465.730,3475.320,1743.460,773.460,2408.560,1695.620,83852.710,168563.040,84906.010,NaT,NaN,NaN,4.000,1.000,1.000,1358.250,4.000,U,A,39.000


In [3]:
customer.describe()

,N_CONTRACTS,FIRST_DCREAT,LAST_DCREAT,LAST_DPOS,LAST_DATFIN,FIRST_D1FIN,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,TOTAL_MENSALIDADE,TOTAL_CRD,TOTAL_SREC,TOTAL_RN,TOTAL_RD,LAST_RISK,MAX_RISKA,MIN_RESSO,MAX_RESSO,MEDIAN_RESSO,CSP,NBENF,EVER_SOL,...,MONTVENC_CP_MIN,MONTVENC_CP_MAX,MONTVENC_CP_MEDIAN,MONTVENC_AUTO_MIN,MONTVENC_AUTO_MAX,MONTVENC_AUTO_MEDIAN,MONTVENC_HT_MIN,MONTVENC_HT_MAX,MONTVENC_HT_MEDIAN,DIVIDAS_CL_MIN,DIVIDAS_CL_MAX,DIVIDAS_CL_MEDIAN,DIVIDAS_CP_MIN,DIVIDAS_CP_MAX,DIVIDAS_CP_MEDIAN,DIVIDAS_AUTO_MIN,DIVIDAS_AUTO_MAX,DIVIDAS_AUTO_MEDIAN,DIVIDAS_HT_MIN,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,sys_data_procura,kp_sqe_enc,ks_score_tier,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_age
count,148729.000,148729,148729,148729,148729,148729,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,148729.000,...,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,141663.000,62173,62173.000,62173.000,141115.000,141115.000,141115.000,141115.000,141115.000,141115.000
mean,1.248,2021-11-26 00:55:05.448163584,2022-05-13 12:16:22.630152704,2024-01-03 17:54:06.363520256,2022-05-22 06:24:52.825205504,2021-12-04 06:36:57.235374080,69.853,74.417,72.129,30.974,35.070,32.889,43.978,44.207,44.110,11521.808,11521.624,262.458,5917.065,-6.465,0.340,1.134,-683574328200651648.000,0.083,1443.922,1493.280,1469.255,62.417,0.612,0.183,...,4.725,207.192,30.687,4.977,83.138,13.782,4.985,147.024,28.034,6795.433,22142.234,10722.432,1248.404,5200.607,2400.276,3112.368,10089.306,5114.468,24906.039,57096.301,29331.216,2024-08-28 00:30:14.910009344,2.152,1.572,8.570,0.782,0.683,438.160,8.570,45.050
min,1.000,1996-06-14 00:00:00,1996-06-26 00:00:00,2014-06-27 00:00:00,1996-07-03 00:00:00,1996-06-14 00:00:00,11.000,11.000,11.000,0.000,0.000,0.000,0.000,0.000,0.000,300.000,300.000,0.621,0.000,-2641.014,0.000,0.000,-9223372036854775808.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2020-08-12 00:00:00,0.000,1.000,1.000,0.000,0.000,-2382.500,1.000,18.000
25%,1.000,2020-06-29 00:00:00,2020-12-22 00:00:00,2023-08-31 00:00:00,2020-12-30 00:00:00,2020-07-07 00:00:00,48.000,48.000,48.000,9.000,14.000,12.000,22.000,22.000,22.000,4900.000,4900.000,115.076,0.000,0.000,0.000,0.000,0.000,0.000,948.935,977.381,963.831,55.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,3500.000,1157.435,0.000,724.565,129.830,0.000,0.000,0.000,0.000,0.000,0.000,2023-12-16 00:00:00,0.000,1.000,4.000,0.000,0.000,-716.750,4.000,36.000
50%,1.000,2022-08-17 00:00:00,2023-04-03 00:00:00,2024-10-02 00:00:00,2023-04-11 00:00:00,2022-08-26 00:00:00,81.000,84.000,84.000,24.000,30.000,27.000,39.000,39.000,39.000,7634.360,7633.110,187.685,2800.762,0.000,0.000,0.000,0.000,0.000,1214.224,1247.041,1230.628,70.000,0.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,2445.450,10934.160,5946.660,195.250,2296.270,1076.080,0.000,1175.990,0.000,0.000,0.000,0.000,2024-10-11 00:00:00,1.000,1.000,7.000,1.000,1.000,253.786,7.000,44.000
75%,1.000,2024-04-06 00:00:00,2024-11-15 00:00:00,2025-07-01 00:00:00,2024-11-22 00:00:00,2024-04-11 00:00:00,84.000,84.000,84.000,48.000,52.000,48.000,61.000,61.000,61.000,15000.000,15000.000,323.692,8179.504,0.000,0.000,0.000,0.000,0.000,1713.806,1747.056,1731.813,80.000,1.000,0.000,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,9121.730,25932.710,14665.720,1300.885,6118.600,3010.030,3265.190,14000.000,8

## 1. Missing values

In [4]:
# Check for columns with >= 30% missing values
null_pct = (customer.isnull().sum() / len(customer) * 100).round(2)
high_missing = null_pct[null_pct >= 30].sort_values(ascending=False)

print(f"Columns with >= 30% missing values: {len(high_missing)}\n")
if len(high_missing) > 0:
    display(high_missing.to_frame("null_%"))
else:
    print("None found. All columns have < 30% missing.")

Columns with >= 30% missing values: 6



,null_%
LAST_OBS_DATE_RBT,97.900
LAST_OBS_DATE_SOL,81.670
LAST_OBS_DATE_SAN,72.700
sys_data_procura,58.200
kp_sqe_enc,58.200
ks_score_tier,58.200


In [5]:
# Drop columns with >= 30% missing values
cols_to_drop = ['LAST_OBS_DATE_RBT', 'LAST_OBS_DATE_SOL', 'LAST_OBS_DATE_SAN',
                'sys_data_procura', 'kp_sqe_enc', 'ks_score_tier']

customer.drop(columns=cols_to_drop, inplace=True)

print(f"Dropped {len(cols_to_drop)} columns: {cols_to_drop}")
print(f"New shape: {customer.shape}")

Dropped 6 columns: ['LAST_OBS_DATE_RBT', 'LAST_OBS_DATE_SOL', 'LAST_OBS_DATE_SAN', 'sys_data_procura', 'kp_sqe_enc', 'ks_score_tier']
New shape: (148729, 82)


## 2. Data Preprocessing Pipeline (encoding + imputation + scalling)

In [6]:
# Identifying categorical category columns
cat_cols = customer.select_dtypes(include=['object', 'category']).columns.tolist()
print(f"Categorical columns ({len(cat_cols)}):\n")
for col in cat_cols:
    print(f"  - {col}  |  unique: {customer[col].nunique()}  |  values: {customer[col].unique()[:10]}")

Categorical columns (3):

  - CONTRIB  |  unique: 148729  |  values: ['00008246f87bcc3c17b90629bb183fe2e58795176310f017217d7749af7ee981'
 '0000ab2116257783438c70ff85a3e98f2d4194ebe534349a33373dfcb3a3a297'
 '0000c74654405ec1da4dbdcd00b86e397954043965d98e542d19fa4808c6b65a'
 '0000e359b15a80ba12c7f60c0fe06ffc68621c87f13a4f88f54188543d9a09c8'
 '0000f858346061c53064586a3347b34659565a6712d004e64309c2473f76faed'
 '00025459b703e1c308553e83a6d545a71fe6a787c2dd1c62d26faa0207cc5b08'
 '000406feeb8088e3b05f47bc89160d25ca14f11c31f91bc6516a4e6753e58d73'
 '00041ebafb1270a818c30cb1fb20d3699002196644ea8fd9df9425df6f49db00'
 '000508d6436a34b780df1aa8068568f1b118f5d589ddc7bb33662295defff2aa'
 '00050fe9f0e69ce221a574af0baaff0b37c598af7a5cc6eb42a2afd52d4e0d4a']
  - sdem_SITFAM  |  unique: 8  |  values: ['S' 'C' None 'U' 'V' 'X' 'F' 'D' 'P']
  - sdem_HABITAT  |  unique: 8  |  values: ['F' 'P' None 'A' 'L' 'X' 'E' 'O' '0']


In [7]:
customer.drop(columns=['CONTRIB'], inplace=True) #unique for each customer, need to drop it now because we won't encode it

In [8]:
date_cols = customer.select_dtypes(include=['datetime64', 'datetimetz']).columns.tolist()
categorical_cols = customer.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = customer.select_dtypes(include=['number']).columns.tolist()

In [9]:
categorical_cols = customer.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = customer.select_dtypes(include=['number']).columns.tolist()
preprocessor = ColumnTransformer([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols),
    ('scaler', RobustScaler(), numerical_cols)
])
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('imputer', IterativeImputer(max_iter=10, random_state=42)),
])
customer_transformed = pipeline.fit_transform(customer)
# Get column names: one-hot encoded + numerical
ohe_cols = pipeline['preprocessor'].named_transformers_['onehot'].get_feature_names_out(categorical_cols).tolist()
all_cols = ohe_cols + numerical_cols
# Convert to DataFrame
customer = pd.DataFrame(customer_transformed, columns=all_cols)

In [10]:
customer.head()

,sdem_SITFAM_C,sdem_SITFAM_D,sdem_SITFAM_F,sdem_SITFAM_P,sdem_SITFAM_S,sdem_SITFAM_U,sdem_SITFAM_V,sdem_SITFAM_X,sdem_SITFAM_None,sdem_HABITAT_0,sdem_HABITAT_A,sdem_HABITAT_E,sdem_HABITAT_F,sdem_HABITAT_L,sdem_HABITAT_O,sdem_HABITAT_P,sdem_HABITAT_X,sdem_HABITAT_None,N_CONTRACTS,MIN_DURDEG,MAX_DURDEG,MEDIAN_DURDEG,MIN_RANGPRO,MAX_RANGPRO,MEDIAN_RANGPRO,MIN_RANGCLI,MAX_RANGCLI,MEDIAN_RANGCLI,TOTAL_MTFIN,TOTAL_MTFINO,...,MONTVENC_CL_MIN,MONTVENC_CL_MAX,MONTVENC_CL_MEDIAN,MONTVENC_CP_MIN,MONTVENC_CP_MAX,MONTVENC_CP_MEDIAN,MONTVENC_AUTO_MIN,MONTVENC_AUTO_MAX,MONTVENC_AUTO_MEDIAN,MONTVENC_HT_MIN,MONTVENC_HT_MAX,MONTVENC_HT_MEDIAN,DIVIDAS_CL_MIN,DIVIDAS_CL_MAX,DIVIDAS_CL_MEDIAN,DIVIDAS_CP_MIN,DIVIDAS_CP_MAX,DIVIDAS_CP_MEDIAN,DIVIDAS_AUTO_MIN,DIVIDAS_AUTO_MAX,DIVIDAS_AUTO_MEDIAN,DIVIDAS_HT_MIN,DIVIDAS_HT_MAX,DIVIDAS_HT_MEDIAN,ALLBD_N_Dossiers__N,ALLBD_A_CL__N,ALLBD_A_CP__N,ALLBD_IDADE_MEAN__N,ALLBD_N_events__N,sdem_age
0,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,1.000,0.083,1.000,0.500,-0.538,-0.447,-0.528,-0.513,-0.513,-0.513,1.620,1.620,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.268,-0.131,-0.440,-0.149,0.261,-0.034,0.000,4.104,1.982,0.000,0.000,0.000,0.429,0.000,1.000,-0.326,0.429,-0.647
1,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,1.083,1.000,1.000,1.718,1.605,1.778,1.333,1.333,1.333,1.224,1.224,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.958,2.622,1.409,4.078,3.518,2.097,0.000,-0.084,0.000,0.000,0.000,0.000,0.429,0.000,-1.000,0.429,0.429,0.471
2,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,1.000,1.000,-0.583,-0.667,-0.667,-0.077,-0.237,-0.167,-0.462,-0.462,-0.462,1.170,1.170,...,182341.845,-80547.216,666711.314,75650.470,-92722.613,317870.383,55232.190,467072.321,221386.059,106574.828,1518881.946,710376.980,-6.429,-125.709,-39.091,2.475,-171.138,-27.794,57.173,-4.609,17.569,-17.653,-30.836,-19.156,0.141,-0.957,-1.396,1.774,0.141,0.582
3,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.250,-0.333,-0.333,0.256,0.105,0.194,-0.128,-0.128,-0.128,-0.508,-0.508,...,-9193.327,4549.159,-33619.889,-3818.437,4902.513,-16033.043,-2787.052,-23518.820,-11174.882,-5380.994,-76609.328,-35872.742,0.825,6.878,2.346,0.723,9.214,1.887,-1.889,0.902,-0.220,1.618,2.389,1.625,-0.571,-1.000,-1.000,0.158,-0.571,1.471
4,0.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.083,0.000,0.000,1.282,1.158,1.306,0.897,0.897,0.897,-0.261,-0.261,...,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.000,-0.156,-0.356,-0.284,0.977,0.219,0.232,0.237,0.088,0.211,2.324,2.332,1.808,-0.429,0.000,0.000,0.519,-0.429,-0.294


In [11]:
total_nulls = np.isnan(customer_transformed).sum()
print(f"Total nulls: {total_nulls}")
if total_nulls == 0:
    print("✅ No missing values found!")
else:
    print(f"⚠️ Still {total_nulls} missing values remaining.")

Total nulls: 0
✅ No missing values found!


## 2. Feature Engineering

### 2.1 Date columns

In [12]:
date_cols = customer.select_dtypes(include=['datetime64', 'datetimetz']).columns.tolist()
print(f"Date columns ({len(date_cols)}):\n")
for col in date_cols:
    print(f"  - {col}")

Date columns (0):



In [13]:
# 1. Ensure all columns are converted to datetime format
date_columns = ['FIRST_DCREAT', 'LAST_DCREAT', 'LAST_DPOS', 'LAST_DATFIN', 'FIRST_D1FIN']
for col in date_columns:
    customer[col] = pd.to_datetime(customer[col], errors='coerce')

# 2. Calculate elapsed time in months using days as the precise base unit
# We use 30.44 as the standard mathematical average for days in a month
avg_days_in_month = 30.44

# Total client tenure (from very first loan to the last event)
customer['client_total_tenure_months'] = (customer['LAST_DPOS'] - customer['FIRST_DCREAT']).dt.days / avg_days_in_month

# Actual duration of the last loan
customer['actual_loan_duration_months'] = (customer['LAST_DPOS'] - customer['LAST_DCREAT']).dt.days / avg_days_in_month

# Months settled early (Theoretical end minus actual event date)
customer['months_settled_early'] = (customer['LAST_DATFIN'] - customer['LAST_DPOS']).dt.days / avg_days_in_month

# Planned duration of the very first loan
customer['first_loan_planned_duration'] = (customer['FIRST_D1FIN'] - customer['FIRST_DCREAT']).dt.days / avg_days_in_month

# 3. Clean up the resulting numeric columns
new_numeric_cols = [
    'client_total_tenure_months', 
    'actual_loan_duration_months', 
    'months_settled_early',
    'first_loan_planned_duration'
]

for col in new_numeric_cols:
    # Round to whole months, fill missing values with 0, and convert to integer
    customer[col] = customer[col].round(0).fillna(0).astype(int)

# Prevent negative values in 'months_settled_early' caused by data entry errors
customer['months_settled_early'] = customer['months_settled_early'].clip(lower=0)

# 4. Drop the original date columns so the K-Means algorithm does not crash
customer = customer.drop(columns=date_columns)

print("All dates transformed successfully into numeric features.")
print(customer[new_numeric_cols].head(30))

KeyError: 'FIRST_DCREAT'

In [ ]:
print(customer[new_numeric_cols].head(10))

   client_total_tenure_months  actual_loan_duration_months  \
0                          17                           14   
1                           1                            1   
2                         153                          153   
3                          47                           47   
4                           0                            0   
5                           0                            0   
6                          84                           84   
7                          46                           46   
8                          13                           13   
9                           0                            0   

   months_settled_early  first_loan_planned_duration  
0                     0                            0  
1                     0                            1  
2                     0                            0  
3                     0                            0  
4                     0                   